# Bitcoin AML Detection | Random Forest Baselines

**Elliptic++ transaction graph · chronological evaluation · four clearly separated models**

This notebook builds four reproducible Random Forest baselines for labeled Bitcoin transactions. It reports illicit-class metrics, compares error tradeoffs, and visualizes results. The goal is a transparent baseline for later comparison with GraphSAGE—not to imply that an alert proves wrongdoing.

> **Model naming:** AF/LF describe the feature family; GNE adds an MLP-based neighborhood embedding. **GNE here is not GraphSAGE.**

**Notebook flow:** setup → data checks and exploration → chronological split → AF → AF + GNE → LF → LF + GNE → evaluation charts → saved artifacts.

## 1 · Imports, paths, and reproducible settings

Run this notebook from the project root or the `Random Forest` folder. The path check locates the project files, selects an existing `data/` or `dataset/` directory, and keeps notebook outputs separate from the source code.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from joblib import dump
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Locate the project root whether VS Code starts Jupyter from the root or this folder.
CURRENT = Path.cwd().resolve()
PROJECT_ROOT = CURRENT.parent if CURRENT.name.lower() == "random forest" else CURRENT
if not (PROJECT_ROOT / "requirements.txt").exists():
    candidates = [p for p in CURRENT.parents if (p / "requirements.txt").exists()]
    if candidates:
        PROJECT_ROOT = candidates[0]

REQUIRED_FILES = ("txs_features.csv", "txs_classes.csv", "txs_edgelist.csv")
def has_transaction_data(folder: Path) -> bool:
    return all((folder / name).is_file() and (folder / name).stat().st_size > 1024 for name in REQUIRED_FILES)

candidates = [PROJECT_ROOT / "data", PROJECT_ROOT / "dataset"]
DATA_DIR = next((folder for folder in candidates if has_transaction_data(folder)), PROJECT_ROOT / "data")
NOTEBOOK_DIR = PROJECT_ROOT / "Random Forest"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_UNTIL = 34
N_TREES = 50
MAX_FEATURES = 50
GNE_HIDDEN_UNITS = 64
GNE_MAX_ITER = 100

# Consistent, presentation-ready chart defaults.
plt.rcParams.update({
    "figure.figsize": (10, 5.5), "figure.dpi": 120,
    "axes.titlesize": 13, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.22,
    "font.size": 10, "legend.frameon": False,
})
MODEL_COLORS = {
    "AF": "#2563eb", "AF + GNE": "#7c3aed",
    "LF": "#0891b2", "LF + GNE": "#ea580c",
}

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Split        : train steps 1–{TRAIN_UNTIL}; test steps {TRAIN_UNTIL + 1}–49")

## 2 · Ensure the transaction dataset is available

The notebook looks for the three transaction CSVs in `data/` and then `dataset/`. If they are missing, it downloads only those files from the public Elliptic++ folder. The large data files and generated outputs should stay out of source control.

In [ ]:
ELLIPTICPP_FOLDER_URL = "https://drive.google.com/drive/folders/1MRPXz79Lu_JGLlJ21MDfML44dKN9R08l"

def ensure_transaction_data(data_dir: Path) -> None:
    data_dir.mkdir(parents=True, exist_ok=True)
    missing = [name for name in REQUIRED_FILES if not (data_dir / name).is_file() or (data_dir / name).stat().st_size <= 1024]
    if not missing:
        print(f"Dataset files ready in {data_dir}")
        return

    print("Dataset files are missing or are Git LFS pointer files; downloading the public transaction CSVs…")
    listing = subprocess.run(
        [sys.executable, "-m", "gdown", ELLIPTICPP_FOLDER_URL, "--json", "--quiet"],
        check=True, capture_output=True, text=True,
    )
    entries = json.loads(listing.stdout)
    available = {Path(item["path"]).name: item["url"] for item in entries if Path(item["path"]).name in REQUIRED_FILES}
    unavailable = set(missing) - set(available)
    if unavailable:
        raise RuntimeError(f"The public folder did not list: {sorted(unavailable)}")
    for name in missing:
        print(f"Downloading {name}…")
        subprocess.run(
            [sys.executable, "-m", "gdown", available[name], "--output", str(data_dir / name), "--continue", "--retries", "3"],
            check=True,
        )

ensure_transaction_data(DATA_DIR)

## 3 · Load features, graph edges, and labels

Elliptic++ labels are mapped as **1 = illicit**, **2 = licit**, **3 = unknown**. Unknown labels stay missing and are excluded from supervised training and scoring; they are not silently relabeled as licit. The feature table’s ID and time columns are metadata, not model inputs.

In [ ]:
feature_path = DATA_DIR / "txs_features.csv"
class_path = DATA_DIR / "txs_classes.csv"
edge_path = DATA_DIR / "txs_edgelist.csv"

feature_columns = pd.read_csv(feature_path, nrows=0).columns.tolist()
if not {"txId", "Time step"}.issubset(feature_columns):
    raise ValueError("Feature CSV must contain txId and Time step columns.")

all_feature_names = [name for name in feature_columns if name not in {"txId", "Time step"}]
local_feature_names = [name for name in all_feature_names if name.startswith("Local_feature_")]
feature_dtypes = {name: np.float32 for name in all_feature_names}
feature_dtypes.update({"txId": np.int64, "Time step": np.int16})
features_df = pd.read_csv(feature_path, dtype=feature_dtypes, low_memory=False)

tx_ids = features_df["txId"].to_numpy(dtype=np.int64)
time_steps = features_df["Time step"].to_numpy(dtype=np.int16)
X_AF = features_df[all_feature_names].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32)
X_LF = features_df[local_feature_names].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=np.float32)
del features_df

classes_df = pd.read_csv(class_path, dtype={"txId": np.int64}, low_memory=False)
if not {"txId", "class"}.issubset(classes_df.columns):
    raise ValueError("Class CSV must contain txId and class columns.")
classes_df["class"] = pd.to_numeric(classes_df["class"], errors="coerce")
raw_labels = classes_df.set_index("txId")["class"].reindex(tx_ids).to_numpy(dtype=np.float32)
y = np.full(len(tx_ids), np.nan, dtype=np.float32)
y[raw_labels == 1] = 1  # illicit = positive class
y[raw_labels == 2] = 0  # licit = negative class

edges_df = pd.read_csv(edge_path, dtype=np.int64, low_memory=False)
if not {"txId1", "txId2"}.issubset(edges_df.columns):
    raise ValueError("Edge CSV must contain txId1 and txId2 columns.")
edges = edges_df[["txId1", "txId2"]].to_numpy(dtype=np.int64)
del edges_df, classes_df, raw_labels

label_counts = pd.Series(y).map({0.0: "Licit", 1.0: "Illicit"}).fillna("Unknown").value_counts()
print(f"Transactions: {len(tx_ids):,} | edges: {len(edges):,}")
print(f"All features: {len(all_feature_names)} | local features: {len(local_feature_names)}")
display(label_counts.rename_axis("Label").to_frame("Transactions"))

## 4 · Explore the label distribution and timeline

These plots show the label imbalance and how labeled/unknown transactions are distributed across time. The class imbalance is why illicit precision, recall, F1, and average precision matter more than accuracy alone.

In [ ]:
labels_for_plot = pd.Series(y).map({0.0: "Licit", 1.0: "Illicit"}).fillna("Unknown").to_numpy()
label_order = ["Licit", "Illicit", "Unknown"]
label_palette = {"Licit": "#2563eb", "Illicit": "#ea580c", "Unknown": "#94a3b8"}
label_totals = pd.Series(labels_for_plot).value_counts().reindex(label_order, fill_value=0)
timeline = pd.DataFrame({"Time step": time_steps, "Label": labels_for_plot}).groupby(["Time step", "Label"]).size().unstack(fill_value=0).reindex(columns=label_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
axes[0].bar(label_totals.index, label_totals.values, color=[label_palette[name] for name in label_totals.index])
axes[0].set_title("Transactions by label")
axes[0].set_ylabel("Number of transactions")
axes[0].ticklabel_format(axis="y", style="plain")
for i, value in enumerate(label_totals.values):
    axes[0].text(i, value, f"{value:,}", ha="center", va="bottom", fontsize=9)

for label in label_order:
    axes[1].plot(timeline.index, timeline[label], marker="o", markersize=3, linewidth=1.8, label=label, color=label_palette[label])
axes[1].axvline(TRAIN_UNTIL + 0.5, color="#111827", linestyle="--", linewidth=1.3, label="Train/test boundary")
axes[1].set_title("Label availability by time step")
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Number of transactions")
axes[1].legend(ncol=2)
plt.show()

## 5 · Define a leakage-aware chronological holdout

Train only on known labels from earlier steps, then test on known labels from later steps. This approximates a future-period evaluation and prevents randomly mixing future transactions into training. It does not eliminate real-world temporal drift.

In [ ]:
labeled_mask = np.isfinite(y)
train_mask = labeled_mask & (time_steps <= TRAIN_UNTIL)
test_mask = labeled_mask & (time_steps > TRAIN_UNTIL)
train_rows = np.flatnonzero(train_mask)
test_rows = np.flatnonzero(test_mask)
y_train = y[train_rows].astype(np.int8)
y_test = y[test_rows].astype(np.int8)

if not len(train_rows) or not len(test_rows):
    raise ValueError("The selected time split has no labeled train or test rows.")
if np.unique(y_train).size != 2 or np.unique(y_test).size != 2:
    raise ValueError("Both classes must appear in the train and test periods.")

split_summary = pd.DataFrame([
    {"Partition": "Train", "Time steps": f"1–{TRAIN_UNTIL}", "Transactions": len(train_rows), "Illicit": int((y_train == 1).sum()), "Licit": int((y_train == 0).sum())},
    {"Partition": "Test", "Time steps": f"{TRAIN_UNTIL + 1}–49", "Transactions": len(test_rows), "Illicit": int((y_test == 1).sum()), "Licit": int((y_test == 0).sum())},
]).set_index("Partition")
display(split_summary.style.format({"Transactions": "{:,.0f}", "Illicit": "{:,.0f}", "Licit": "{:,.0f}"}).background_gradient(cmap="Blues", subset=["Transactions"]))

RF_SETTINGS = dict(n_estimators=N_TREES, max_features=MAX_FEATURES, random_state=SEED, n_jobs=-1, class_weight=None)
results = {}
models = {}
predictions = {}

def fit_evaluate_save(model_name: str, feature_matrix: np.ndarray, feature_names: list[str] | None = None) -> None:
    # Fit one forest, score illicit as positive, and save its model/predictions.
    forest = RandomForestClassifier(**{**RF_SETTINGS, "max_features": min(MAX_FEATURES, feature_matrix.shape[1])})
    forest.fit(feature_matrix[train_rows], y_train)
    predicted = forest.predict(feature_matrix[test_rows]).astype(np.int8)
    positive_col = int(np.flatnonzero(forest.classes_ == 1)[0])
    probability = forest.predict_proba(feature_matrix[test_rows])[:, positive_col]
    cm = confusion_matrix(y_test, predicted, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    results[model_name] = {
        "accuracy": float(accuracy_score(y_test, predicted)),
        "precision_illicit": float(precision_score(y_test, predicted, zero_division=0)),
        "recall_illicit": float(recall_score(y_test, predicted, zero_division=0)),
        "f1_illicit": float(f1_score(y_test, predicted, zero_division=0)),
        "average_precision_illicit": float(average_precision_score(y_test, probability)),
        "roc_auc_illicit": float(roc_auc_score(y_test, probability)),
        "specificity_licit": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "balanced_accuracy": float(balanced_accuracy_score(y_test, predicted)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "test_rows": int(len(test_rows)),
        "feature_count": int(feature_matrix.shape[1]),
    }
    predictions[model_name] = {"labels": predicted, "probability": probability}
    models[model_name] = forest
    dump(forest, MODEL_DIR / f"{model_name.replace(' ', '_')}.joblib")
    pd.DataFrame({
        "txId": tx_ids[test_rows], "Time step": time_steps[test_rows],
        "true_label_0_licit_1_illicit": y_test,
        "predicted_label_0_licit_1_illicit": predicted,
        "illicit_probability": probability,
    }).to_csv(OUTPUT_DIR / f"predictions_{model_name.replace(' ', '_')}.csv", index=False)
    print(f"{model_name}: Accuracy {results[model_name]['accuracy']:.4f} | Illicit P {results[model_name]['precision_illicit']:.4f} | R {results[model_name]['recall_illicit']:.4f} | F1 {results[model_name]['f1_illicit']:.4f}")
    display(pd.DataFrame(classification_report(y_test, predicted, labels=[0, 1], target_names=["Licit", "Illicit"], output_dict=True, zero_division=0)).T.round(4))

print(f"Scoring {len(test_rows):,} held-out known-label transactions.")

# Model A · Random Forest AF

**Inputs:** all numeric transaction features. This is the all-features tabular baseline, with no neighborhood embedding.

In [ ]:
fit_evaluate_save("AF", X_AF, all_feature_names)

# Model B · Random Forest AF + GNE

**Inputs:** all transaction features plus a 64-value learned neighborhood representation. One-hop feature means include only neighbors from the same or an earlier time step. The embedding is produced by a small MLP trained on training-period labels; it is a project approximation, not GraphSAGE.

In [ ]:
def make_causal_neighbor_inputs(X: np.ndarray, ids: np.ndarray, steps: np.ndarray, edge_pairs: np.ndarray) -> np.ndarray:
    # Append one-hop neighbor means without using later-time neighbors.
    id_to_row = {int(tx_id): row for row, tx_id in enumerate(ids)}
    left = np.fromiter((id_to_row.get(int(value), -1) for value in edge_pairs[:, 0]), dtype=np.int64, count=len(edge_pairs))
    right = np.fromiter((id_to_row.get(int(value), -1) for value in edge_pairs[:, 1]), dtype=np.int64, count=len(edge_pairs))
    keep = (left >= 0) & (right >= 0)
    left, right = left[keep], right[keep]
    src = np.concatenate((left, right))
    dst = np.concatenate((right, left))
    causal = steps[src] <= steps[dst]
    src, dst = src[causal], dst[causal]
    sums = np.zeros_like(X, dtype=np.float32)
    counts = np.zeros(X.shape[0], dtype=np.float32)
    np.add.at(sums, dst, X[src])
    np.add.at(counts, dst, 1.0)
    has_neighbors = counts > 0
    sums[has_neighbors] /= counts[has_neighbors, None]
    return np.concatenate((X, sums), axis=1)

def fit_gne(neighbor_features: np.ndarray, fit_rows: np.ndarray | None = None) -> tuple[np.ndarray, StandardScaler, MLPClassifier]:
    # Fit only on the explicitly supplied labeled rows; default to the full training period.
    fit_rows = train_rows if fit_rows is None else fit_rows
    scaler = StandardScaler().fit(neighbor_features[fit_rows])
    scaled = scaler.transform(neighbor_features).astype(np.float32)
    encoder = MLPClassifier(
        hidden_layer_sizes=(GNE_HIDDEN_UNITS,), activation="relu", solver="adam",
        alpha=1e-4, batch_size=256, learning_rate_init=1e-3,
        max_iter=GNE_MAX_ITER, early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=10, random_state=SEED, verbose=False,
    )
    encoder.fit(scaled[fit_rows], y[fit_rows].astype(np.int8))
    hidden = np.maximum(scaled @ encoder.coefs_[0] + encoder.intercepts_[0], 0).astype(np.float32)
    return hidden, scaler, encoder

X_AF_neighbors = make_causal_neighbor_inputs(X_AF, tx_ids, time_steps, edges)
AF_embedding, AF_scaler, AF_encoder = fit_gne(X_AF_neighbors)
X_AF_GNE = np.concatenate((X_AF, AF_embedding), axis=1)
fit_evaluate_save("AF + GNE", X_AF_GNE)
dump({"scaler": AF_scaler, "encoder": AF_encoder, "feature_family": "AF"}, MODEL_DIR / "AF_GNE_encoder.joblib")
del X_AF_neighbors, AF_embedding, X_AF_GNE

# Model C · Random Forest LF

**Inputs:** only columns named `Local_feature_*`. This tests a narrower feature family without graph-neighborhood context.

In [ ]:
fit_evaluate_save("LF", X_LF, local_feature_names)

# Model D · Random Forest LF + GNE

**Inputs:** local features plus an MLP-based embedding of local features and causal one-hop neighbors. It lets us compare neighborhood context within the same local-only feature family.

In [ ]:
X_LF_neighbors = make_causal_neighbor_inputs(X_LF, tx_ids, time_steps, edges)
LF_embedding, LF_scaler, LF_encoder = fit_gne(X_LF_neighbors)
X_LF_GNE = np.concatenate((X_LF, LF_embedding), axis=1)
fit_evaluate_save("LF + GNE", X_LF_GNE)
dump({"scaler": LF_scaler, "encoder": LF_encoder, "feature_family": "LF"}, MODEL_DIR / "LF_GNE_encoder.joblib")
del X_LF_neighbors, LF_embedding, X_LF_GNE

## 10 · Compare the four models

The table reports both threshold-dependent detection metrics and ranking metrics. `AF + GNE` and `LF + GNE` are evaluated with the same temporal split and held-out IDs as their feature-only counterparts.

In [ ]:
MODEL_ORDER = ["AF", "AF + GNE", "LF", "LF + GNE"]
metrics_df = pd.DataFrame(results).T.reindex(MODEL_ORDER)
display(metrics_df.style.format({column: "{:.4f}" for column in metrics_df.select_dtypes(include="number").columns if column not in {"tn", "fp", "fn", "tp", "test_rows", "feature_count"}})
        .background_gradient(cmap="Blues", subset=["f1_illicit", "average_precision_illicit"])
        .set_caption("Held-out model metrics · illicit is the positive class"))

metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index_label="model")
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Metrics and predictions saved to: {OUTPUT_DIR}")

In [ ]:
primary_metrics = ["accuracy", "precision_illicit", "recall_illicit", "f1_illicit"]
ranking_metrics = ["average_precision_illicit", "roc_auc_illicit", "balanced_accuracy"]
x = np.arange(len(MODEL_ORDER))
width = 0.18
metric_colors = ["#64748b", "#2563eb", "#ea580c", "#7c3aed"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)
for index, metric in enumerate(primary_metrics):
    axes[0].bar(x + (index - 1.5) * width, metrics_df.loc[MODEL_ORDER, metric], width, label=metric.replace("_illicit", "").replace("_", " ").title(), color=metric_colors[index])
axes[0].set_title("Classification metrics at the default decision rule")
axes[0].set_ylabel("Score")
axes[0].set_xticks(x, MODEL_ORDER)
axes[0].set_ylim(0, 1.08)
axes[0].legend(ncol=2, fontsize=8)

for index, metric in enumerate(ranking_metrics):
    axes[1].bar(x + (index - 1) * 0.22, metrics_df.loc[MODEL_ORDER, metric], 0.22, label=metric.replace("_illicit", "").replace("_", " ").upper(), color=["#0891b2", "#7c3aed", "#16a34a"][index])
axes[1].set_title("Ranking and class-balance metrics")
axes[1].set_ylabel("Score")
axes[1].set_xticks(x, MODEL_ORDER)
axes[1].set_ylim(0, 1.08)
axes[1].legend(fontsize=8)
fig.suptitle("Random Forest baseline comparison", fontsize=16, fontweight="bold")
plt.show()

## 11 · Inspect false alerts and missed illicit transactions

Each confusion matrix uses rows = actual `[licit, illicit]` and columns = predicted `[licit, illicit]`. The normalized view makes class-specific error rates easier to compare when the licit class is much larger.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for ax, model_name in zip(axes.flat, MODEL_ORDER):
    p = predictions[model_name]["labels"]
    cm = confusion_matrix(y_test, p, labels=[0, 1])
    row_totals = cm.sum(axis=1, keepdims=True)
    normalized = np.divide(cm, row_totals, out=np.zeros_like(cm, dtype=float), where=row_totals != 0)
    image = ax.imshow(normalized, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(model_name, color=MODEL_COLORS[model_name], fontweight="bold")
    ax.set_xticks([0, 1], ["Licit", "Illicit"])
    ax.set_yticks([0, 1], ["Licit", "Illicit"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    for row in range(2):
        for col in range(2):
            text_color = "white" if normalized[row, col] > 0.55 else "#0f172a"
            ax.text(col, row, f"{cm[row, col]:,}\n{normalized[row, col]:.1%}", ha="center", va="center", color=text_color, fontsize=10)
fig.colorbar(image, ax=axes, shrink=0.8, label="Share of actual class")
fig.suptitle("Error profile by model", fontsize=16, fontweight="bold")
plt.show()

## 12 · Compare ROC and precision–recall curves

Curves evaluate model probabilities across thresholds. For rare illicit cases, read the precision–recall curve and average precision together with ROC-AUC; ROC-AUC alone can look optimistic under class imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for model_name in MODEL_ORDER:
    probability = predictions[model_name]["probability"]
    fpr, tpr, _ = roc_curve(y_test, probability)
    precision, recall, _ = precision_recall_curve(y_test, probability)
    axes[0].plot(fpr, tpr, linewidth=2, color=MODEL_COLORS[model_name], label=f"{model_name} · AUC {results[model_name]['roc_auc_illicit']:.3f}")
    axes[1].plot(recall, precision, linewidth=2, color=MODEL_COLORS[model_name], label=f"{model_name} · AP {results[model_name]['average_precision_illicit']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="#94a3b8", linewidth=1, label="Random ranking")
prevalence = float(y_test.mean())
axes[1].axhline(prevalence, linestyle="--", color="#94a3b8", linewidth=1, label=f"Test prevalence {prevalence:.1%}")
axes[0].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.02))
axes[1].set(title="Precision–recall curve", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
fig.suptitle("Threshold-independent model behavior", fontsize=16, fontweight="bold")
plt.show()

## 13 · Which features drive the feature-only forests?

Random Forest impurity importance provides a quick descriptive view, not causal evidence. Correlated features can share or distort importance; use permutation importance or SHAP with care for a more rigorous explanation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, name, names in [(axes[0], "AF", all_feature_names), (axes[1], "LF", local_feature_names)]:
    importance = pd.Series(models[name].feature_importances_, index=names).nlargest(12).sort_values()
    ax.barh(importance.index, importance.values, color=MODEL_COLORS[name])
    ax.set_title(f"{name} · top 12 impurity importances", color=MODEL_COLORS[name], fontweight="bold")
    ax.set_xlabel("Mean decrease in impurity")
fig.suptitle("Feature importance for feature-only models", fontsize=16, fontweight="bold")
plt.show()

## 14 · Summary and saved artifacts

The following cell identifies the leading Random Forest variants by illicit F1 and recall. It deliberately avoids claiming that any model is superior to GraphSAGE: that comparison requires training GraphSAGE on this same data version and held-out transaction set.

In [ ]:
best_f1_model = metrics_df["f1_illicit"].idxmax()
best_recall_model = metrics_df["recall_illicit"].idxmax()
best_ap_model = metrics_df["average_precision_illicit"].idxmax()
display(Markdown(
    f"**Best Random Forest illicit F1:** {best_f1_model} ({metrics_df.loc[best_f1_model, 'f1_illicit']:.4f})  \n"
    f"**Best illicit recall:** {best_recall_model} ({metrics_df.loc[best_recall_model, 'recall_illicit']:.4f})  \n"
    f"**Best average precision:** {best_ap_model} ({metrics_df.loc[best_ap_model, 'average_precision_illicit']:.4f})  \n\n"
    f"Artifacts saved under `{OUTPUT_DIR}`. These models rank transactions for review; their predictions are not findings of criminal conduct."
))
print("Saved model files:")
for path in sorted(MODEL_DIR.glob("*.joblib")):
    print(f"  • {path.name}")

## Evaluation notes and fair GraphSAGE comparison

- Keep the data version, label mapping, train/test time cutoff, and exact test transaction IDs fixed across models.
- Tune GraphSAGE parameters and its decision threshold using validation data from training-period transactions only; do not tune on the final test labels.
- Compare illicit precision, recall, F1, average precision, ROC-AUC, and confusion matrices. Report multiple random seeds and training cost when feasible.
- Unknown labels are omitted from this supervised benchmark; this means the reported metrics apply only to the known-label subset.
- The graph-neighborhood embedding in this notebook is an MLP-based approximation. It must not be presented as a GraphSAGE implementation.

## 15 · Optional validation-driven tuning experiment

This experiment compares the baseline forests with validation-selected class weighting and decision thresholds. **Settings are selected only on a chronological validation period** (steps 28–34); the final test period (steps 35–49) is used only for reporting. The notebook’s four original baseline sections remain unchanged.

Threshold tuning can trade precision for recall. It does not guarantee that all three improve, especially when transaction behavior shifts over time. Keep the untuned baseline as the reference unless the untouched holdout confirms a useful gain.

In [ ]:
# Hold out steps 28–34 from model fitting while choosing class weights and thresholds.
VALIDATION_START = TRAIN_UNTIL - 7  # train 1–27; validate 28–34 for the default cutoff of 34
selection_train_rows = np.flatnonzero(labeled_mask & (time_steps <= VALIDATION_START))
validation_rows = np.flatnonzero(labeled_mask & (time_steps > VALIDATION_START) & (time_steps <= TRAIN_UNTIL))
y_selection_train = y[selection_train_rows].astype(np.int8)
y_validation = y[validation_rows].astype(np.int8)

if np.unique(y_selection_train).size != 2 or np.unique(y_validation).size != 2:
    raise ValueError("The tuning split must contain both known classes in train and validation periods.")

TUNED_OUTPUT_DIR = OUTPUT_DIR / "tuned"
TUNED_MODEL_DIR = TUNED_OUTPUT_DIR / "models"
TUNED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
class_weight_options = [None, "balanced_subsample"]
tuned_settings = {}
tuned_results = {}
tuned_predictions = {}

print(f"Tuning fit rows: {len(selection_train_rows):,}; validation rows: {len(validation_rows):,}; final holdout rows: {len(test_rows):,}.")

In [ ]:
def select_rf_settings(model_name: str, feature_matrix: np.ndarray) -> dict:
    # Choose class weight and threshold that maximize validation illicit F1.
    candidates = []
    for class_weight in class_weight_options:
        candidate = RandomForestClassifier(
            **{**RF_SETTINGS, "max_features": min(MAX_FEATURES, feature_matrix.shape[1]), "class_weight": class_weight}
        )
        candidate.fit(feature_matrix[selection_train_rows], y_selection_train)
        positive_column = int(np.flatnonzero(candidate.classes_ == 1)[0])
        validation_probability = candidate.predict_proba(feature_matrix[validation_rows])[:, positive_column]
        precision, recall, thresholds = precision_recall_curve(y_validation, validation_probability)
        f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        best_f1 = float(f1_values.max())
        tied = np.flatnonzero(np.isclose(f1_values, best_f1))
        best_index = max(tied, key=lambda index: (precision[index], thresholds[index]))
        candidates.append({
            "class_weight": class_weight,
            "threshold": float(thresholds[best_index]),
            "validation_precision": float(precision[best_index]),
            "validation_recall": float(recall[best_index]),
            "validation_f1": float(f1_values[best_index]),
            "validation_average_precision": float(average_precision_score(y_validation, validation_probability)),
        })
    chosen = max(candidates, key=lambda item: (item["validation_f1"], item["validation_precision"], item["validation_recall"]))
    print(f"{model_name}: selected {chosen}")
    return chosen


def fit_tuned_final(model_name: str, feature_matrix: np.ndarray, settings: dict, encoder_bundle: dict | None = None) -> dict:
    # Refit on the full training period and evaluate once on the later holdout.
    forest = RandomForestClassifier(
        **{**RF_SETTINGS, "max_features": min(MAX_FEATURES, feature_matrix.shape[1]), "class_weight": settings["class_weight"]}
    )
    forest.fit(feature_matrix[train_rows], y_train)
    positive_column = int(np.flatnonzero(forest.classes_ == 1)[0])
    probability = forest.predict_proba(feature_matrix[test_rows])[:, positive_column]
    predicted = (probability >= settings["threshold"]).astype(np.int8)
    cm = confusion_matrix(y_test, predicted, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    summary = {
        "accuracy": float(accuracy_score(y_test, predicted)),
        "precision_illicit": float(precision_score(y_test, predicted, zero_division=0)),
        "recall_illicit": float(recall_score(y_test, predicted, zero_division=0)),
        "f1_illicit": float(f1_score(y_test, predicted, zero_division=0)),
        "average_precision_illicit": float(average_precision_score(y_test, probability)),
        "roc_auc_illicit": float(roc_auc_score(y_test, probability)),
        "threshold": float(settings["threshold"]),
        "class_weight": settings["class_weight"],
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    key = model_name.replace(" + ", "+")
    tuned_results[model_name] = summary
    tuned_predictions[model_name] = {"labels": predicted, "probability": probability}
    dump(forest, TUNED_MODEL_DIR / f"{key}.joblib")
    if encoder_bundle is not None:
        dump(encoder_bundle, TUNED_MODEL_DIR / f"{key}_encoder.joblib")
    pd.DataFrame({
        "txId": tx_ids[test_rows], "Time step": time_steps[test_rows],
        "true_label_0_licit_1_illicit": y_test,
        "predicted_label_0_licit_1_illicit": predicted,
        "illicit_probability": probability,
    }).to_csv(TUNED_OUTPUT_DIR / f"predictions_{key}.csv", index=False)
    print(f"{model_name} holdout: P={summary['precision_illicit']:.4f} R={summary['recall_illicit']:.4f} F1={summary['f1_illicit']:.4f}")
    return summary

### Tuning A · AF

Fit candidate forests on steps 1–27, select class weight and threshold on steps 28–34, then refit the selected configuration on steps 1–34.

In [ ]:
tuned_settings["AF"] = select_rf_settings("AF", X_AF)
fit_tuned_final("AF", X_AF, tuned_settings["AF"])

### Tuning B · AF + GNE

The validation MLP is fitted only on steps 1–27. After selecting Random Forest settings, a fresh MLP is fitted on steps 1–34 before the final held-out evaluation.

In [ ]:
AF_neighbor_features = make_causal_neighbor_inputs(X_AF, tx_ids, time_steps, edges)
AF_validation_embedding, _, _ = fit_gne(AF_neighbor_features, fit_rows=selection_train_rows)
AF_validation_matrix = np.concatenate((X_AF, AF_validation_embedding), axis=1)
tuned_settings["AF + GNE"] = select_rf_settings("AF + GNE", AF_validation_matrix)
del AF_validation_embedding, AF_validation_matrix

AF_final_embedding, AF_final_scaler, AF_final_encoder = fit_gne(AF_neighbor_features, fit_rows=train_rows)
AF_final_matrix = np.concatenate((X_AF, AF_final_embedding), axis=1)
fit_tuned_final("AF + GNE", AF_final_matrix, tuned_settings["AF + GNE"], {"scaler": AF_final_scaler, "encoder": AF_final_encoder, "feature_family": "AF"})
del AF_neighbor_features, AF_final_embedding, AF_final_matrix, AF_final_scaler, AF_final_encoder

### Tuning C · LF

Use only local features for candidate selection and the final refit.

In [ ]:
tuned_settings["LF"] = select_rf_settings("LF", X_LF)
fit_tuned_final("LF", X_LF, tuned_settings["LF"])

### Tuning D · LF + GNE

This repeats the same validation-only selection and full-training-period refit for local features plus neighborhood context.

In [ ]:
LF_neighbor_features = make_causal_neighbor_inputs(X_LF, tx_ids, time_steps, edges)
LF_validation_embedding, _, _ = fit_gne(LF_neighbor_features, fit_rows=selection_train_rows)
LF_validation_matrix = np.concatenate((X_LF, LF_validation_embedding), axis=1)
tuned_settings["LF + GNE"] = select_rf_settings("LF + GNE", LF_validation_matrix)
del LF_validation_embedding, LF_validation_matrix

LF_final_embedding, LF_final_scaler, LF_final_encoder = fit_gne(LF_neighbor_features, fit_rows=train_rows)
LF_final_matrix = np.concatenate((X_LF, LF_final_embedding), axis=1)
fit_tuned_final("LF + GNE", LF_final_matrix, tuned_settings["LF + GNE"], {"scaler": LF_final_scaler, "encoder": LF_final_encoder, "feature_family": "LF"})
del LF_neighbor_features, LF_final_embedding, LF_final_matrix, LF_final_scaler, LF_final_encoder

## 16 · Baseline versus validation-tuned results

The table and chart use the same test transactions for both sets of predictions. A useful tuning change should improve the selected metric on this later period, not just on validation. Threshold movement usually shifts precision and recall in opposite directions; compare F1 and the confusion matrix before deciding which setting is appropriate.

In [ ]:
tuned_metrics_df = pd.DataFrame(tuned_results).T.reindex(MODEL_ORDER)
comparison_rows = []
for model_name in MODEL_ORDER:
    for version, frame in [("Baseline", metrics_df), ("Validation-tuned", tuned_metrics_df)]:
        comparison_rows.append({
            "Model": model_name,
            "Version": version,
            "Precision": float(frame.loc[model_name, "precision_illicit"]),
            "Recall": float(frame.loc[model_name, "recall_illicit"]),
            "F1": float(frame.loc[model_name, "f1_illicit"]),
        })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.pivot(index="Model", columns="Version", values=["Precision", "Recall", "F1"]).round(4))

TUNED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tuned_metrics_df.to_csv(TUNED_OUTPUT_DIR / "metrics_tuned.csv", index_label="model")
(TUNED_OUTPUT_DIR / "metrics_tuned.json").write_text(json.dumps({"settings": tuned_settings, "holdout": tuned_results}, indent=2), encoding="utf-8")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
for ax, metric, label in zip(axes, ["precision_illicit", "recall_illicit", "f1_illicit"], ["Illicit precision", "Illicit recall", "Illicit F1"]):
    positions = np.arange(len(MODEL_ORDER))
    width = 0.34
    baseline_values = metrics_df.loc[MODEL_ORDER, metric].astype(float).to_numpy()
    tuned_values = tuned_metrics_df.loc[MODEL_ORDER, metric].astype(float).to_numpy()
    ax.bar(positions - width / 2, baseline_values, width, color=[MODEL_COLORS[name] for name in MODEL_ORDER], label="Baseline")
    ax.bar(positions + width / 2, tuned_values, width, color=[MODEL_COLORS[name] for name in MODEL_ORDER], alpha=0.48, hatch="//", label="Validation-tuned")
    ax.set_title(label)
    ax.set_xticks(positions, MODEL_ORDER, rotation=12)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
fig.suptitle("Did validation-based tuning improve the later-period holdout?", fontsize=15, fontweight="bold")
plt.show()

display(Markdown(
    "**Read the result honestly:** settings are selected on steps 28–34 only. "
    "If holdout F1 falls while recall rises, the model has shifted toward more alerts and more false positives. "
    "Do not describe that as a blanket improvement; state which error tradeoff the application values."
))
print(f"Tuned artifacts saved to: {TUNED_OUTPUT_DIR}")

## Viva takeaway on tuning

I tested class weighting (`None` versus `balanced_subsample`) and a validation-selected probability threshold. For this chronological split, validation performance does not guarantee improvement on later transactions because the transaction distribution changes over time. Report the baseline and tuned holdout metrics side by side, explain the precision–recall tradeoff, and do not claim improvement unless the held-out results support it.

## Expanded walk-forward tuning check

To test whether the first threshold search was too narrow, I also compared four Random Forest configurations using two expanding validation windows inside training steps 1–34: train 1–27 / validate 28–30, then train 1–30 / validate 31–34. I pooled only those validation predictions to select an illicit-class F1 threshold. The small grid varied tree count, feature sampling, minimum leaf size, and class weighting. GNE encoders were refit inside each fold using that fold's training rows.

All four feature variants selected the same forest configuration: 100 trees, `max_features="sqrt"`, `min_samples_leaf=2`, and `class_weight="balanced_subsample"`. The thresholds and later-period scores were:

| Model | Threshold | Accuracy | Illicit precision | Illicit recall | Illicit F1 | Average precision |
|---|---:|---:|---:|---:|---:|---:|
| AF | 0.350 | 0.9732 | 0.8370 | 0.7304 | 0.7801 | 0.7917 |
| AF + GNE | 0.353 | 0.9790 | 0.9432 | 0.7202 | 0.8168 | 0.7893 |
| LF | 0.455 | 0.9712 | 0.8080 | 0.7304 | 0.7672 | 0.7857 |
| LF + GNE | 0.420 | 0.9727 | 0.8319 | 0.7267 | 0.7758 | 0.7873 |

This is a sensitivity analysis, not a clean final test: steps 35–49 had already been examined in earlier iterations. It did not improve F1 over the previous baseline run, though recall and ranking average precision were stronger in the saved trial metrics. The tuned thresholds traded some precision for recall, so keep the default-threshold baseline as the primary result unless the project explicitly prioritizes finding more illicit examples over generating fewer false alerts. Report all metrics together and do not claim one threshold maximizes precision, recall, and F1 at once.


To reproduce this expanded search from the project root, run `python src/walk_forward_rf_tuning.py`. It saves the fold selections and exploratory later-period metrics under `outputs/tuning/`.
